In [1]:
#r "D:/rieckmann/BoSSS/BoSSS-Repos/BoSSS-InterfaceSlip/public/src/L4-application/BoSSSpad/bin/Release/net6.0/BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
string proj = "CapillaryWaves3";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


Project name is set to 'CapillaryWaves3'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\backup\rieckmann\cluster\databases\CapillaryWaves3'.


In [3]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [4]:
int[] kelemSeq = new int[]{3};
var GridSeq = new IGridInfo[kelemSeq.Length];

In [5]:
for(int iGrid = 0; iGrid < GridSeq.Length; iGrid++) {
    
    int kelem = kelemSeq[iGrid];
    
    GridCommons grd;       
    double[] xNodes = GenericBlas.Linspace(-1.0, 1.0, 2);
    double[] yNodes = GenericBlas.Linspace(-1.0, 1.0, 2); 
    double h = (yNodes.Last()-yNodes.First())/kelem;    
    var box1 = new GridCommons.GridBox(new double[] {xNodes.First(), yNodes.First()}, new double[] {xNodes.Last(), yNodes.Last()}, kelem, kelem);
    var box2 = new GridCommons.GridBox(new double[] {xNodes.First(), yNodes.First()+h}, new double[] {xNodes.Last(), yNodes.Last()-h}, 3*kelem, kelem);
    var box3 = new GridCommons.GridBox(new double[] {xNodes.First(), yNodes.First()+h+h/3}, new double[] {xNodes.Last(), yNodes.Last()-h-h/3}, 3*3*kelem, kelem);

    grd = Grid2D.HangingNodes2D(true, false, box1, box2, box3);

    // HIER ANPASSUNG DER RANDBEDINGUNGEN!
    grd.EdgeTagNames.Add(1, "pressure_outlet");

 
    grd.DefineEdgeTags(delegate (double[] X) {
        byte et = 0;
        if (Math.Abs(X[1] - yNodes.Last()) <= 1.0e-8) // UPPER BOUNDARY
            et = 1;
        if (Math.Abs(X[1] - yNodes.First()) <= 1.0e-8) // LOWER BOUNDARY
            et = 1;
        if (Math.Abs(X[0] - xNodes.First()) <= 1.0e-8) // LEFT BOUNDARY
            et = 1;
        if (Math.Abs(X[0] - xNodes.Last()) <= 1.0e-8) // RIGHT BOUNDARY
            et = 1;
        return et;
    });

    // grd.Name = "StaticDropletOnWall_meshStudy";
    grd.Name = "CapillaryWaves_meshstudy_" + kelem;        
    
    BoSSSshell.WorkflowMgm.DefaultDatabase.SaveGrid(ref grd);
    
    GridSeq[iGrid] = grd;
}

Grid Edge Tags changed.
An equivalent grid (c8ffda49-d74c-4ee7-bfa6-4eaf4f895e7a) is already present in the database -- the grid will not be saved.


In [26]:
public static XNSE_Control CW_forWorksheet() {

    XNSE_Control C = new XNSE_Control();

    //C.DbPath = set by workflowMgm during job creation
    C.savetodb = true;
    C.ContinueOnIoError = false;
    
    // Physical Parameters
    // ===================
    C.PhysicalParameters.IncludeConvection = false;
    C.PhysicalParameters.Material = false;

    // misc. solver options
    // ====================
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    // Level-Set options (AMR)
    // =======================
    C.LSContiProjectionMethod = BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.ConstrainedDG;
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    // Timestepping
    // ============
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C.Timestepper_BDFinit = BoSSS.Solution.Timestepping.TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;

    C.Endtime = 1000;
    C.saveperiod = 1;       

    return C;

}

public static List<XNSE_Control> CapillaryWave(int[] kS, double[] tlvls, int[] pDegS, IGridInfo[] grdS){
    
    List<XNSE_Control> controls = new List<XNSE_Control>();
    foreach(int k in kS){
        foreach(double tlvl in tlvls){
            foreach(int pDeg in pDegS){
                foreach(IGridInfo grd in grdS){
                    controls.Add(CapillaryWave(k, tlvl, pDeg, grd));
                }
            }
        }
    }
    
    return controls;
    
}

public static XNSE_Control CapillaryWave(int k, double tlvl, int pDeg, IGridInfo grd) {
    
    var C = CW_forWorksheet(); // ICH HABE DIE WESENTLICHEN EINSTELLUNGEN HIER DIREKT IN DAS WORKSHEET ALS STATISCHE METHODE GESCHRIEBEN!

    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("wavenumber", k));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("degree", pDeg));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("gridlevel", ((GridCommons)grd).Name.Split('_').Last()));

    C.SetDGdegree(pDeg);        
    C.SetGrid(grd);

    C.PhysicalParameters.rho_A             = 1.0;
    C.PhysicalParameters.rho_B             = 1.0;
    C.PhysicalParameters.mu_A              = 0.0001;
    C.PhysicalParameters.mu_B              = 0.0001;
    C.PhysicalParameters.Sigma             = 1.0;

    C.LinearSolver                  = LinearSolverCode.direct_mumps.GetConfig();
    // C.LinearSolver                  = LinearSolverCode.direct_pardiso.GetConfig();
    
    //VERDAMPFUNG IN MEDIUM A
    C.AddBoundaryValue("pressure_outlet");

//     C.AddInitialValue("Phi", $"X => X[1] - 0.1*Math.Exp(-(X[0]*X[0])/(0.1*0.1))", false); // planar with a smoothed dirac to exite some capillary waves
    C.AddInitialValue("Phi", $"X => X[1] - 0.01*Math.Sin(2*Math.PI*{k}*X[0])", false); // standing wave as an initial disturbance

    // MAKE TIMESTEPPING SETTINGS MORE CLEAR!
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    
    // set timestep, such that the grid N=17 should just obey the capillary timestep restriction:
    double c = Math.Sqrt((2*Math.PI*k*C.PhysicalParameters.Sigma)/(C.PhysicalParameters.rho_A+C.PhysicalParameters.rho_B)); // (undamped) phasevelocity of the standing wave 
    double dt = ((GridCommons)grd).GridData.Cells.h_minGlobal /((2*pDeg+1)*c);
    C.dtFixed = dt * Math.Pow(2.0, tlvl);
    C.Endtime = 2.0/c;
    String.Format($"c: {c}; dt: {C.dtFixed}; t1: {C.Endtime}").Display();
    C.NoOfTimesteps = (int)Math.Ceiling(C.Endtime/C.dtFixed);
    
    C.SessionName = ((GridCommons)grd).Name + //grid
        "_P" + pDeg + // degree 
        "_DT" + C.dtFixed.ToString("N5"); // timestep 
    
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("timelevel", tlvl.ToString("N2")));
    
    C.TracingNamespaces = "*";
    
    return C;
}

In [27]:
int[] kS = new int[] { 1 };
int[] degS = new int[] { 2 };
double[] tlvlS = new double[] {-0.5, 0, 0.5};

In [28]:
int iDeg0 = 0;
int iGrd0 = 0;

In [29]:
List<XNSE_Control> controls = new List<XNSE_Control>();

In [30]:
// Testcase setup
// ==============

{
    var ctrls = CapillaryWave(kS, tlvlS, degS,GridSeq);
    controls.AddRange(ctrls);
}

Console.WriteLine("# of controls: " + controls.Count());


c: 1.7724538509055159; dt: 0.005910256005947143; t1: 1.1283791670955126

c: 1.7724538509055159; dt: 0.008358364200707489; t1: 1.1283791670955126

c: 1.7724538509055159; dt: 0.011820512011894286; t1: 1.1283791670955126

# of controls: 3


In [31]:
controls.ForEach(c => Console.WriteLine(c.SessionName));

CapillaryWaves_meshstudy_3_P2_DT0.00591
CapillaryWaves_meshstudy_3_P2_DT0.00836
CapillaryWaves_meshstudy_3_P2_DT0.01182


In [33]:
// // FOR TESTING IF THIS RUNS
// foreach(var C in controls){
var C = controls.ElementAt(0);
C.ImmediatePlotPeriod = 1;
C.SuperSampling = 2;
C.savetodb = false;
C.PostprocessingModules.Clear();
using(var solver = new XNSE()){
    solver.Init(C);
    solver.RunSolverMode();
}
// }


Session ID: 00000000-0000-0000-0000-000000000000, DB path: '\\dc3\backup\rieckmann\cluster\databases\CapillaryWaves3'
Session directory '\\dc3\backup\rieckmann\cluster\databases\CapillaryWaves3\sessions\00000000-0000-0000-0000-000000000000'.
Grid repartitioning method: METIS
Grid repartitioning options: 
Number of cell Weights: 0
Going with agglomeration threshold: 0.1
Linearization hint: AdHoc
=============== Operator Configuration ===============
     isGravity                     :[ ]
     isVolForce                    :[ ]
     isTransport                   :[ ]
     isViscous                     :[x]
     isPressureGradient            :[x]
     isInterfaceSlip               :[ ]
     isContinuity                  :[x]
     isMovingMesh                  :[ ]
     isMatInt                      :[x]
     isPInterfaceSet               :[ ]
     isImmersedBoundary            :[ ]
     withPressureDissipation       :[ ]
=============== Linear Solver Configuration ===============
     So

    23,    23,     2	1.350768E-014       1.444021E-014       6.022664E-015       2.067001E-014       
Done with time step 23; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 24, dt = 0.005910256005947143 ...
    24,    24,     2	1.002799E-014       1.107584E-014       2.322455E-014       2.761548E-014       
Done with time step 24; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 25, dt = 0.005910256005947143 ...
    25,    25,     2	1.020506E-014       1.319570E-014       6.890812E-015       1.804863E-014       
Done with time step 25; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 26, dt = 0.005910256005947143 ...
    26,    26,     2	1.082444E-014       1.248513E-014       4.061738E-014       4.384995E-014       
Done with 

Starting time step 51, dt = 0.005910256005947143 ...
    51,    51,     2	6.783486E-015       3.920071E-014       9.608627E-015       4.092721E-014       
Done with time step 51; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 52, dt = 0.005910256005947143 ...
    52,    52,     2	8.321502E-015       4.223230E-014       6.948601E-015       4.360158E-014       
Done with time step 52; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 53, dt = 0.005910256005947143 ...
    53,    53,     2	1.047220E-014       3.128705E-014       2.231042E-015       3.306848E-014       
Done with time step 53; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 54, dt = 0.005910256005947143 ...
    54,    54,     2	1.615164E-014       7.926861E-014     

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 79, dt = 0.005910256005947143 ...
    79,    79,     2	8.670472E-015       1.125160E-014       2.131549E-015       1.436381E-014       
Done with time step 79; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 80, dt = 0.005910256005947143 ...
    80,    80,     2	9.858601E-015       1.069383E-014       1.405857E-014       2.022853E-014       
Done with time step 80; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 81, dt = 0.005910256005947143 ...
    81,    81,     2	8.474403E-015       1.060520E-014       2.855184E-015       1.387220E-014       
Done with time step 81; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time ste

   106,   106,     2	2.885245E-014       1.751558E-013       4.470783E-015       1.775725E-013       
Done with time step 106; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 107, dt = 0.005910256005947143 ...
   107,   107,     2	2.834477E-014       1.178721E-013       9.940701E-015       1.216391E-013       
Done with time step 107; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 108, dt = 0.005910256005947143 ...
   108,   108,     2	2.268620E-014       1.735709E-013       3.176590E-015       1.750760E-013       
Done with time step 108; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 109, dt = 0.005910256005947143 ...
   109,   109,     2	6.164728E-014       3.240518E-013       1.807077E-014       3.303581E-013       
Done


All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 134, dt = 0.005910256005947143 ...
   134,   134,     2	2.895341E-014       1.463263E-013       6.675635E-015       1.493126E-013       
Done with time step 134; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 135, dt = 0.005910256005947143 ...
   135,   135,     2	1.830285E-014       1.442402E-013       1.895336E-014       1.466269E-013       
Done with time step 135; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 136, dt = 0.005910256005947143 ...
   136,   136,     2	1.874378E-014       1.198720E-013       6.114039E-015       1.214825E-013       
Done with time step 136; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting t

   161,   161,     2	8.824943E-015       1.144793E-014       8.889384E-015       1.696926E-014       
Done with time step 161; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 162, dt = 0.005910256005947143 ...
   162,   162,     2	9.166614E-015       1.078884E-014       3.373547E-015       1.455358E-014       
Done with time step 162; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 163, dt = 0.005910256005947143 ...
   163,   163,     2	8.008024E-015       9.254605E-015       5.899904E-015       1.358621E-014       
Done with time step 163; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 164, dt = 0.005910256005947143 ...
   164,   164,     2	8.047678E-015       8.239434E-015       1.338883E-014       1.766109E-014       
Done


All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 189, dt = 0.005910256005947143 ...
   189,   189,     2	9.803076E-015       3.424780E-014       7.015247E-015       3.630738E-014       
Done with time step 189; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 190, dt = 0.005910256005947143 ...
   190,   190,     2	1.106295E-014       4.992365E-014       1.207247E-014       5.254050E-014       
Done with time step 190; solver success: True

All Cells: min=105 max=105 avg=105 inb=0 tot=105
Cut Cells: min=81 max=81 avg=81 inb=0, tot=81
Starting time step 191, dt = 0.005910256005947143 ...
   191,   191,     2	9.635681E-015       4.151768E-014       8.758854E-015       4.351186E-014       
Done with time step 191; solver success: True

Removing tag: NotTerminated



In [15]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();

In [16]:
bpc

RuntimeLocation,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,\\dc3\backup\rieckmann\cluster\binaries,True,DefaultDC2,dotnet,FDY\rieckmann,DC2,"[ hpccluster, hpccluster2, hpcluster3 ]",Normal,True,List<AllowedDatabasesPair> \\dc3\backup\rieckmann\cluster\databases


In [17]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();

In [18]:
jobs.ActivateTasks((MsHPC2012Client)bpc);

Deploying executables and additional files...
   copied 'win\amd64' runtime.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


...

   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj



In [19]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00219	02/01/2024 17:26:45	8d6a749e...
#1: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00438	02/01/2024 17:26:45	1f8c9dfa...
#2: CapillaryWaves2	CapillaryWaves_meshstudy_3_P3_DT0.00071*	02/01/2024 17:26:45	578d61fd...
#3: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00110*	02/01/2024 17:26:45	86d9ae8c...
#4: CapillaryWaves2	CapillaryWaves_meshstudy_3_P3_DT0.00142*	02/01/2024 17:26:45	abad0b16...
#5: CapillaryWaves2	CapillaryWaves_meshstudy_3_P4_DT0.00204*	02/01/2024 17:26:45	0d8c844b...
#6: CapillaryWaves2	CapillaryWaves_meshstudy_3_P3_DT0.00285*	02/01/2024 17:26:45	19c4f939...
#7: CapillaryWaves2	CapillaryWaves_meshstudy_3_P4_DT0.00815*	02/01/2024 17:26:45	1d5e1557...
#8: CapillaryWaves2	CapillaryWaves_meshstudy_3_P4_DT0.00407*	02/01/2024 17:26:45	378b3b5a...
#9: CapillaryWaves2	CapillaryWaves_meshstudy_3_P3_DT0.01138*	02/01/2024 17:26:45	5d1b8bed...
#10: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.01753*	02/01/2024

In [20]:
//var sess = sessions.Where(s => s.SuccessfulTermination);
var sess = sessions.Where(s => Convert.ToInt32(s.KeysAndQueries["id:degree"]) == 2);
sess

#0: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00219	02/01/2024 17:26:45	8d6a749e...
#1: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00438	02/01/2024 17:26:45	1f8c9dfa...
#2: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00110*	02/01/2024 17:26:45	86d9ae8c...
#3: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.01753*	02/01/2024 17:26:45	9e289fd6...
#4: CapillaryWaves2	CapillaryWaves_meshstudy_3_P2_DT0.00876*	02/01/2024 17:26:45	e352210c...


In [21]:
sess.ForEach(s => s.Export().To(Path.GetFullPath(Path.Join(".",BoSSSshell.WorkflowMgm.CurrentProject + "-" + s.Name))).WithShadowFields().WithSupersampling(2).Do());

Starting export process... Data will be written to the directory: D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\examples\SlipConvergence\CapillaryWaves2-CapillaryWaves_meshstudy_3_P2_DT0.00219
Starting export process... Data will be written to the directory: D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\examples\SlipConvergence\CapillaryWaves2-CapillaryWaves_meshstudy_3_P2_DT0.00438
Starting export process... Data will be written to the directory: D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\examples\SlipConvergence\CapillaryWaves2-CapillaryWaves_meshstudy_3_P2_DT0.00110
Starting export process... Data will be written to the directory: D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\examples\SlipConvergence\CapillaryWaves2-CapillaryWaves_meshstudy_3_P2_DT0.01753
Starting export process... Data will be written to the directory: D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\examples\SlipConvergence\CapillaryWaves2-CapillaryWaves_